# 第 1 周末练习 —— DevOps / 云基础设施技术问答

## 练习目标（理念）

为了熟悉 **提示词（prompts）**、**messages** 结构以及 **OpenAI Chat Completions API**，请构建一个小工具：

- **输入**：一个技术问题（例如 PromQL 告警、K8s、CI/CD）
- **输出**：清晰、简洁、偏生产实践的回答
- **实现方式**：用 `openai` 库发起一次（或多次）`chat.completions.create`

## 和本课 Day 1 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| `messages`（system / user） | `create_prompt()` 拼出列表 |
| Chat Completions API | `make_api_call()` → `openai.chat.completions.create` |
| system 定角色、user 放问题 | DevOps 助手 + PromQL 提问 |
| 环境变量 | `OPENAI_API_KEY`、可选 `OPENAI_BASE_URL` |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 准备好 `.env`：至少有 `OPENAI_API_KEY`；若走代理/兼容端点再设 `OPENAI_BASE_URL`
3. 改写 `user_prompt`（或 `system_prompt`）再跑调用格，对比回答


In [13]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 API Key、Base URL
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 【注】本练习未使用网页抓取；若扩展成「抓站再问答」可取消下一行注释
# from scraper import fetch_website_contents
# 从 IPython.display 导入展示工具：在笔记本里漂亮地显示 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：调用云端（或兼容端点）Chat Completions API
from openai import OpenAI

# 加载 .env：把 OPENAI_API_KEY 等读入进程环境（不写进笔记本正文）
load_dotenv()


True

In [14]:
# ========== 常量：模型名与密钥/端点集中写在一处 ==========

# 选用的云端小模型：便宜、够用，适合做解释类问答
MODEL = 'gpt-4o-mini'
# 从环境变量读取 OpenAI API Key（常见名字 OPENAI_API_KEY）
API_KEY = os.getenv('OPENAI_API_KEY')
# 可选：自定义 Base URL（代理、Azure/兼容网关、本地 OpenAI 兼容服务等）
BASE_URL = os.getenv('OPENAI_BASE_URL')


In [19]:
# ========== 创建 OpenAI 客户端：后续所有 chat 调用都走它 ==========

# api_key / base_url 来自上格常量；base_url 为 None 时 SDK 会用官方默认端点
openai = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
)


In [16]:
# ========== 四个小函数：拼 messages → 调 API → 打印 / Markdown 展示 ==========

# 把 system + user 拼成 OpenAI 期望的 messages 列表（role/content 字典）
def create_prompt(system_prompt, user_prompt):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]  

# 发起一次非流式 Chat Completions，取出助手回复的纯文本
def make_api_call(messages):
    # model / messages 决定「用谁答、答什么」；返回值里 choices[0] 是第一条候选
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages
    )
    # message.content：助手生成的完整字符串
    return response.choices[0].message.content

# 用普通 print 把结果打到 stdout（便于对照原始文本）
def print_result(result):
    print(result)

# 用 IPython Markdown 渲染结果（标题、列表、代码块会更好看）
def display_result(result):
    display(Markdown(result))


In [17]:
# ========== 提问 + 调用：system 定 DevOps 角色，user 放具体 PromQL 问题 ==========

# system_prompt 保留英文：这是发给模型的角色指令，改译会改变回答风格/行为
system_prompt = """
You are a DevOps and Cloud Infrastructure assistant.

Help with:
- Kubernetes, Docker, AWS, and Terraform
- CI/CD pipelines and automation
- Monitoring with Prometheus and Grafana
- PostgreSQL and system performance troubleshooting
- Basic AI infrastructure and LLM deployment

Provide clear, practical, and production-ready answers.
Use examples when helpful.
Avoid unnecessary explanations.
Assume the user has technical experience.
"""

# user_prompt 保留英文：真正的用户问题；改这里就能换题重跑
user_prompt = """
Give me a PromQL query to monitor PostgreSQL CPU usage and alert if it exceeds 80% for 5 minutes.
"""

# 拼出 messages：system + user
messages = create_prompt(system_prompt, user_prompt)

# 调 API，拿到助手文本
result = make_api_call(messages)

# 先用 print 看一遍原始回复
print_result(result)



You can use the following PromQL query to monitor the CPU usage of PostgreSQL and set up an alert for when it exceeds 80% for 5 minutes.

```yaml
alert: PostgreSQLHighCPUUsage
  expr: avg by(instance)(rate(node_cpu_seconds_total{mode="idle"}[5m])) < 0.2
  for: 5m
  labels:
    severity: critical
  annotations:
    summary: "High CPU usage detected on PostgreSQL instance"
    description: "CPU usage is above 80% for the last 5 minutes."
```

### Explanation:
- `node_cpu_seconds_total`: This metric comes from the Prometheus Node Exporter and counts the total time spent by CPU in various modes.
- `mode="idle"`: We filter for idle time to determine the active CPU usage.
- `rate(...[5m])`: This calculates the per-second average rate of CPU time over the last 5 minutes.
- `avg by(instance)`: This ensures we are looking at the average CPU usage per instance.
- `< 0.2`: This checks if the idle CPU percentage is less than 20%, indicating more than 80% usage.

Add this alert rule to your Prometh

In [18]:
# ========== 再用 Markdown 漂亮展示同一份 result ==========

# display_result：把上格得到的 result 渲染成 Markdown（需先跑完上格）
display_result(result)


You can use the following PromQL query to monitor the CPU usage of PostgreSQL and set up an alert for when it exceeds 80% for 5 minutes.

```yaml
alert: PostgreSQLHighCPUUsage
  expr: avg by(instance)(rate(node_cpu_seconds_total{mode="idle"}[5m])) < 0.2
  for: 5m
  labels:
    severity: critical
  annotations:
    summary: "High CPU usage detected on PostgreSQL instance"
    description: "CPU usage is above 80% for the last 5 minutes."
```

### Explanation:
- `node_cpu_seconds_total`: This metric comes from the Prometheus Node Exporter and counts the total time spent by CPU in various modes.
- `mode="idle"`: We filter for idle time to determine the active CPU usage.
- `rate(...[5m])`: This calculates the per-second average rate of CPU time over the last 5 minutes.
- `avg by(instance)`: This ensures we are looking at the average CPU usage per instance.
- `< 0.2`: This checks if the idle CPU percentage is less than 20%, indicating more than 80% usage.

Add this alert rule to your Prometheus alerting rules file and reload the configuration for it to take effect.